# Bayesian Optimization using OPTIMEO


In [ ]:
# For Google Colab
!pip install git+https://github.com/colinbousige/OPTIMEO.git# For local development, prefer:
# uv venv .venv --python 3.10 && source .venv/bin/activate && uv sync


## Single outcome
### First, we generate some data

Let's create an `experimental_data(temp, conc)` function that simulates the yield of a chemical reaction based on temperature and concentration. The yield is a function of these two variables, and we will use Bayesian Optimization to find the position of the maximum in the minimum number of experiments.

In the following block, we just plot this function to see what it looks like(but in real life, you have no idea what the function looks like). We also create an experimental data set where we have already done some experiments (they are the red crosses in the plot). 


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
# pio.renderers.default = "notebook"
pio.renderers.default = "colab"

def experimental_data(temp, conc):
    """
    This function simulates experimental data based on temperature and concentration.
    The function is a Gaussian-like function that peaks at certain temperature and concentration values.
    The function is not based on any real experimental data and is purely for demonstration purposes.
    """
    out = np.exp(-((temp - 50) ** 2)/1000)*np.exp(-((conc - .50) ** 2)/.05)-.9*np.exp(-((temp - 45) ** 2)/100)*np.exp(-((conc - .450) ** 2)/.05)
    return out

def generate_data(N=100):
    temp = np.linspace(0, 100, N)
    conc = np.linspace(0, 1, N)
    data = np.meshgrid(temp, conc)
    temp, conc = data[0].flatten(), data[1].flatten()
    exp_data = experimental_data(temp, conc)
    df = pd.DataFrame({'Temperature': temp, 'Concentration': conc, 'Yield': exp_data})
    return df

df = generate_data(500)

# Prepare data for 3D surface plot
unique_temps = np.unique(df['Temperature'])
unique_concs = np.unique(df['Concentration'])
Z = df.pivot(index='Concentration', columns='Temperature', values='Yield').values
# find the maximum yield and its position
max_yield = np.max(Z)
max_pos = np.unravel_index(np.argmax(Z), Z.shape)
max_temp = unique_temps[max_pos[1]]
max_conc = unique_concs[max_pos[0]]

# Create the contour plot
fig = go.Figure(data=[go.Contour(
    z=Z,
    x=unique_temps,
    y=unique_concs,
    colorscale='Viridis',
    contours=dict(coloring='heatmap', size=0.1),
    hovertemplate='Temperature: %{x}<br>Concentration: %{y}<br>Yield: %{z}<extra></extra>'
)])

# Update layout with legend at the top, below the title
fig.update_layout(
    xaxis_title='Temperature',
    yaxis_title='Concentration',
    legend=dict(
        x=0.5,  # Center the legend horizontally
        y=1.05,  # Place the legend just below the title
        orientation='h',  # Horizontal orientation
        xanchor='center',
        yanchor='bottom'
    )
)

# Add a marker for the maximum yield
fig.add_trace(go.Scatter(
    x=[max_temp],
    y=[max_conc],
    mode='markers',
    marker=dict(size=10, color='red', symbol='x'),
    name='Max Yield that we want to determine through experiments',
    customdata=[[max_yield]],  # Add max_yield as custom data
    hovertemplate='Max Yield:<br>Temperature: %{x}<br>Concentration: %{y}<br>Yield: %{customdata[0]}<extra></extra>'
))

# Create some sample data
df_sample = generate_data(2)

# Add the experimental data on the plot
fig.add_trace(go.Scatter(
    x=df_sample['Temperature'],
    y=df_sample['Concentration'],
    mode='markers',
    marker=dict(size=10, color='white', symbol='circle',
                line=dict(width=2, color='black')),
    name='Experimental measurements we have so far',
))

fig.show()

# Save the experimental data to a CSV file
df_sample.to_csv('experimental_data.csv', index=False)

## Bayesian Optimization

Now, we will use the OPTIMEO package to perform Bayesian Optimization.
First, we load the data in the right format.

In [ ]:
from optimeo.bo import BOExperiment, read_experimental_data

# there is only one outcome here, in the last column (-1)
features, outcomes = read_experimental_data('experimental_data.csv', out_pos=[-1])
print(f"Features:\n{features}")
print(f"Outcomes:\n{outcomes}")

The ranges and types of the features are automatically determined from the data (see the data printed above). In case you want to change the ranges or types, you can do so editing the corresponding fields in the `features` dict or by providing a `ranges` dict. The ranges should be in the format `{'feature_name': [minvalue,maxvalue]}`. If the ranges are not provided either in `features` or in `ranges`, they will be determined from the data.

In [ ]:
ranges = {'Temperature': [-10,100]}

Now, let's create the BOExperiment object. This object contains the data, the features, and the model. The model is a Gaussian Process with a Matern kernel.

In [ ]:
bo = BOExperiment(
    features=features, 
    outcomes=outcomes,
    # ranges=ranges,
    N = 1, # number of new points to generate
    maximize=True, # we want to maximize the response
    outcome_constraints=None,
    fixed_features=None, # fixed features are not used here
    # but they can be added as fixed_features = {Temperature: 50, Concentration: 0.5}
    feature_constraints=None, # feature constraints are not used here
    # but they can be added as 
    # feature_constraints = ['Concentration + Temperature <= 200']
    optim = 'sobol', # sobol is used to randomly generate the new points
    # to actually optimize, use optim = 'bo'
)

In [ ]:
bo

In [ ]:
new_points = bo.suggest_next_trials()
print(f"New points to sample:\n{new_points}")
fig = bo.plot_model()
fig.show()

Now let's do the optimization. We will first perform 6 random point generations, then use Bayesian Optimization to find the maximum of the function.By default, OPTIMEO uses a balanced acquisition setup. You can also tune exploration vs exploitation with UCB-style acquisition settings via `acq_func`:- For single candidate generation (`N = 1`): use `UpperConfidenceBound`- For batch generation (`N > 1`): use `qUpperConfidenceBound``beta` controls exploration: higher = more exploration, lower = more exploitation.

In [ ]:
for i in range(30): #let's do 30 iterations
    if i==6:
        bo.optim = 'bo' # change to BO optimization after 6 iterations
    # simulate the new points
    new = bo.suggest_next_trials(with_predicted=False)
    newT = new['Temperature'].values
    newC = new['Concentration'].values
    # perform an experiment to measure the response at these points
    # here we just simulate the response using the experimental data function
    # in a real experiment, you would measure the response at these points
    # and add the new points to the experimental data
    measured_yield = experimental_data(newT, newC)
    # add the new points to the experimental data
    bo.update_experiment(params   = {'Temperature':newT, 'Concentration':newC}, 
                         outcomes = {'Yield': measured_yield})

Now le'ts plot the model:

In [ ]:
bo.plot_model()

In [ ]:
print(f"Best parameters from BO:")
print(bo.get_best_parameters())
print(f"Expected best parameters:")
print(pd.DataFrame({'Temperature': max_temp, 'Concentration': max_conc, 'Yield':max_yield}, index=[0]))

And the convergence plot: we see here that the maximum was found after the 13th iteration. You can play on the number of random points to see how it affects the convergence.

In [ ]:
bo.plot_optimization_trace(optimum=max_yield)

## Two outcomes
### First, we generate some data

Like before, we will generate some data. We will use the same function for the yield, and andd another function for the price of the experiment: we will want to maximize rthe yield and minimize the price. The price is a function of the temperature and concentration, but it is not a function of the yield. 

In [ ]:
from plotly.subplots import make_subplots

def price(temp, conc):
    """
    This function simulates the price of the experiment.
    """
    out = (np.exp(-((temp - 45) ** 2)/2000)*np.exp(-((conc - .350) ** 2)/.08)-1.2*np.exp(-((temp - 55) ** 2)/150)*np.exp(-((conc - .250) ** 2)/.05))*100+150
    return out

def generate_data(N=100):
    temp = np.linspace(0, 100, N)
    conc = np.linspace(0, 1, N)
    data = np.meshgrid(temp, conc)
    temp, conc = data[0].flatten(), data[1].flatten()
    exp_data = experimental_data(temp, conc)
    price_data = price(temp, conc)
    df = pd.DataFrame({'Temperature': temp, 'Concentration': conc, 'Yield': exp_data, 'Price': price_data})
    return df

df = generate_data(500)

# Prepare data for 3D surface plot
unique_temps = np.unique(df['Temperature'])
unique_concs = np.unique(df['Concentration'])
Z = df.pivot(index='Concentration', columns='Temperature', values='Yield').values
ZZ = df.pivot(index='Concentration', columns='Temperature', values='Price').values
# find the maximum yield and its position
max_yield = np.max(Z)
max_pos = np.unravel_index(np.argmax(Z), Z.shape)
max_temp = unique_temps[max_pos[1]]
max_conc = unique_concs[max_pos[0]]
# min price
min_price = np.min(ZZ)
min_pos = np.unravel_index(np.argmin(ZZ), ZZ.shape)
minp_temp = unique_temps[min_pos[1]]
minp_conc = unique_concs[min_pos[0]]

# Create the subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=('Yield', 'Price'))

# Add the contour plot for Yield (Z)
fig.add_trace(
    go.Contour(
        z=Z,
        x=unique_temps,
        y=unique_concs,
        colorscale='Viridis',
        contours=dict(coloring='heatmap', size=0.1),
        hovertemplate='Temperature: %{x}<br>Concentration: %{y}<br>Yield: %{z}<extra></extra>'
    ),
    row=1, col=1
)

# Add the contour plot for Price (ZZ)
fig.add_trace(
    go.Contour(
        z=ZZ,
        x=unique_temps,
        y=unique_concs,
        colorscale='Viridis',
        contours=dict(coloring='heatmap', size=0.1),
        hovertemplate='Temperature: %{x}<br>Concentration: %{y}<br>Price: %{z}<extra></extra>'
    ),
    row=1, col=2
)

# Update layout with legend at the top, below the title
fig.update_layout(
    xaxis_title='Temperature',
    yaxis_title='Concentration',
    xaxis2_title='Temperature',
    yaxis2_title='Concentration',
    legend=dict(
        x=0.5,  # Center the legend horizontally
        y=1.05,  # Place the legend just below the title
        orientation='h',  # Horizontal orientation
        xanchor='center',
        yanchor='bottom'
    )
)

# Add a marker for the maximum yield
fig.add_trace(
    go.Scatter(
        x=[max_temp],
        y=[max_conc],
        mode='markers',
        marker=dict(size=10, color='red', symbol='x'),
        name='Max Yield',
        customdata=[[max_yield]],  # Add max_yield as custom data
        hovertemplate='Max Yield:<br>Temperature: %{x}<br>Concentration: %{y}<br>Yield: %{customdata[0]}<extra></extra>'
    ),
    row=1, col=1
)

# Add a marker for the minimum price
fig.add_trace(
    go.Scatter(
        x=[minp_temp],
        y=[minp_conc],
        mode='markers',
        marker=dict(size=10, color='orange', symbol='x'),
        name='Min Price',
        customdata=[[min_price]],  # Add min_price as custom data
        hovertemplate='Min Price:<br>Temperature: %{x}<br>Concentration: %{y}<br>Price: %{customdata[0]}<extra></extra>'
    ),
    row=1, col=2
)

# Create some sample data
df_sample = generate_data(2)

# Add the experimental data on the plot for Yield
fig.add_trace(
    go.Scatter(
        x=df_sample['Temperature'],
        y=df_sample['Concentration'],
        mode='markers',
        marker=dict(size=10, color='white', symbol='circle',
                    line=dict(width=2, color='black')),
        name='Experimental Measurements (Yield)',
    ),
    row=1, col=1
)

# Add the experimental data on the plot for Price
fig.add_trace(
    go.Scatter(
        x=df_sample['Temperature'],
        y=df_sample['Concentration'],
        mode='markers',
        marker=dict(size=10, color='white', symbol='circle',
                    line=dict(width=2, color='black')),
        name='Experimental Measurements (Price)',
    ),
    row=1, col=2
)

fig.show()

df_sample.to_csv('experimental_data2.csv', index=False)

Let's now read the experimental data and create our BOExperiment object. The data is in the same format as before, but we have two outcomes: yield and price. The features are the same as before, but we have to specify that we have two outcomes. We also have to specify the type of optimization: we want to maximize the yield and minimize the price.

In [ ]:
features, outcomes = read_experimental_data('experimental_data2.csv', out_pos=[-2,-1])
print(f"- Features:\n{features}")
print(f"- Outcomes:\n{outcomes}")

In [ ]:
bo = BOExperiment(
    features=features, 
    outcomes=outcomes,
    N = 1, # number of new points to generate
    maximize={'Yield':True, 'Price':False}, # we want to maximize the response
    outcome_constraints=None,
    fixed_features=None, # fixed features are not used here
    # but they can be added as fixed_features = {Temperature: 50, Concentration: 0.5}
    feature_constraints=None, # feature constraints are not used here
    # but they can be added as 
    # feature_constraints = ['Concentration + Temperature <= 200']
    optim = 'sobol', # sobol is used to randomly generate the new points
)

Let's do the optimization:

In [ ]:
for i in range(50): #let's do 50 iterations
    if i==6:
        bo.optim = 'bo' # change to BO optimization after 6 iterations
    # simulate the new points
    new = bo.suggest_next_trials(with_predicted=False)
    newT = new['Temperature'].values
    newC = new['Concentration'].values
    # perform an experiment to measure the response at these points
    # here we just simulate the response using the experimental data function
    # in a real experiment, you would measure the response at these points
    # and add the new points to the experimental data
    measured_yield = experimental_data(newT, newC)
    measured_price = price(newT, newC)
    # add the new points to the experimental data
    bo.update_experiment(params   = {'Temperature':newT, 'Concentration':newC}, 
                         outcomes = {'Yield': measured_yield, 'Price': measured_price})

Now let's plot the model:

In [ ]:
figs = [bo.plot_model(metricname=mname) for mname in bo.out_names]
for fig in figs:
    fig.show()

The `get_best_parameters()` function returns the best parameters found so far. In the case of multiple outcomes, it will be an ensemble of points.

In [ ]:
bo.get_best_parameters()

You can also plot the Pareto frontier, and then decide what is the best compromise you are prepared to make between the two outcomes. The Pareto frontier is the set of points that are not dominated by any other point. A point is dominated if there is another point that is better in both outcomes.

Here, you basically see that there are two choices for you, either high yield and high price, or low yield and low price.

In [ ]:
bo.plot_pareto_frontier()